# client-stdio Deep Dive
### Watching the JSON-RPC conversation between client and server

Run each cell top to bottom. By the end you'll see the **raw JSON-RPC messages** flying between the client and the MCP server — the actual protocol that makes plug-and-play possible.

---
## Cell 1 — Imports & Patch

`nest_asyncio` is required to run `asyncio` inside Jupyter without getting a *"This event loop is already running"* error.  
`subprocess` is needed so we can pass `subprocess.DEVNULL` as the `errlog` — Jupyter's fake stderr doesn't have a real file descriptor, which would otherwise crash MCP's Windows process spawner.

In [6]:
import asyncio
import subprocess
import json
import nest_asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from mcp.types import TextContent
from anyio import EndOfStream, ClosedResourceError

nest_asyncio.apply()  # Needed to run async code in Jupyter

print("✅ Imports loaded. nest_asyncio applied — async code can now run in Jupyter.")

✅ Imports loaded. nest_asyncio applied — async code can now run in Jupyter.


---
## Cell 2 — Server Parameters

This tells the MCP client **how to spawn the server** as a subprocess on your machine.  
The client will run `python server.py` in the background over **stdio** (stdin/stdout pipes).  
No HTTP port, no network — just pipes between two processes.

In [7]:
server_params = StdioServerParameters(
    command="python",   # The executable to run
    args=["server.py"], # Arguments — this is your MCP server script
)

print(f"📦 Server params defined.")
print(f"   Command : {server_params.command}")
print(f"   Args    : {server_params.args}")
print("\n   When we open the client, 'python server.py' will be spawned as a subprocess.")
print("   The client and server will talk to each other over stdin/stdout pipes (stdio).")

📦 Server params defined.
   Command : python
   Args    : ['server.py']

   When we open the client, 'python server.py' will be spawned as a subprocess.
   The client and server will talk to each other over stdin/stdout pipes (stdio).


---
## Cell 3 — Stream Wrapper Classes (The Wiretap)

`stdio_client` gives us two raw streams:
- `read_stream`  — messages coming **from** the server
- `write_stream` — messages going **to** the server

We wrap both to **intercept and print every message** before passing it through to `ClientSession`.  
This lets us see the actual JSON-RPC protocol in action without changing any MCP internals.

In [8]:
class LoggingReadStream:
    def __init__(self, stream):
        self._stream = stream
        
    # Add context manager support
    async def __aenter__(self):
        return self
        
    async def __aexit__(self, exc_type, exc_val, exc_tb):
        pass
        
    async def receive(self):
        msg = await self._stream.receive()
        # Ensure we only log if it's the right message type
        if hasattr(msg, "message"):
            print(f"📥 SERVER → CLIENT:\n{json.dumps(msg.message.model_dump(exclude_none=True), indent=2)}\n")
        return msg

    def __aiter__(self):
        return self

    async def __anext__(self):
        try:
            # Delegate to our receive method so logging still happens
            return await self.receive()
        except (EndOfStream, ClosedResourceError):
            # Tell the async loop we have reached the end of the stream
            raise StopAsyncIteration

class LoggingWriteStream:
    def __init__(self, stream):
        self._stream = stream
    # Add context manager support
    async def __aenter__(self):
        return self
    async def __aexit__(self, exc_type, exc_val, exc_tb):
        pass
    async def send(self, msg):
        # Access the wrapped JSON-RPC message
        if hasattr(msg, "message"):
            print(f"📤 CLIENT → SERVER:\n{json.dumps(msg.message.model_dump(exclude_none=True), indent=2)}\n")
        await self._stream.send(msg)


print("✅ Stream wrapper classes defined.")
print("   LoggingReadStream  → prints every message the server sends us.")
print("   LoggingWriteStream → prints every message we send to the server.")

✅ Stream wrapper classes defined.
   LoggingReadStream  → prints every message the server sends us.
   LoggingWriteStream → prints every message we send to the server.


---
## Cell 4 — Phase 1: The Handshake (`session.initialize()`)

This is the **capability negotiation** step — the very first thing that happens after the connection opens.  

Watch the JSON-RPC closely:
- The client sends an `initialize` request advertising what **it** supports
- The server responds with what **it** supports (tools, resources, prompts, etc.)
- The client sends an `notifications/initialized` to confirm — handshake complete

This is the MCP equivalent of a USB device saying *"I am a keyboard, here is what I can do."*

In [9]:
async def phase1_handshake():
    # Define server parameters inside or ensure they are accessible
    server_params = StdioServerParameters(
        command="python",
        args=["server.py"]
    )
    
    async with stdio_client(server_params, errlog=subprocess.DEVNULL) as (read, write):
        logged_read  = LoggingReadStream(read)
        logged_write = LoggingWriteStream(write)
        async with ClientSession(logged_read, logged_write) as session:
            print("🔌 Connection opened — server subprocess is running.")
            print("=" * 60)
            print("HANDSHAKE: session.initialize()")
            print("=" * 60 + "\n")
            result = await session.initialize()
            print("=" * 60)
            print(f"✅ Handshake complete!")
            print(f"   Server name    : {result.serverInfo.name}")
            print(f"   Server version : {result.serverInfo.version}")
            print(f"   MCP version    : {result.protocolVersion}")
            print("=" * 60)
asyncio.run(phase1_handshake())

🔌 Connection opened — server subprocess is running.
HANDSHAKE: session.initialize()

📤 CLIENT → SERVER:
{
  "method": "initialize",
  "params": {
    "protocolVersion": "2025-11-25",
    "capabilities": {},
    "clientInfo": {
      "name": "mcp",
      "version": "0.1.0"
    }
  },
  "jsonrpc": "2.0",
  "id": 0
}

📥 SERVER → CLIENT:
{
  "jsonrpc": "2.0",
  "id": 0,
  "result": {
    "protocolVersion": "2025-11-25",
    "capabilities": {
      "experimental": {},
      "prompts": {
        "listChanged": false
      },
      "resources": {
        "subscribe": false,
        "listChanged": false
      },
      "tools": {
        "listChanged": false
      }
    },
    "serverInfo": {
      "name": "Calculator",
      "version": "1.27.0"
    }
  }
}

📤 CLIENT → SERVER:
{
  "method": "notifications/initialized",
  "jsonrpc": "2.0"
}

✅ Handshake complete!
   Server name    : Calculator
   Server version : 1.27.0
   MCP version    : 2025-11-25


---
## Cell 5 — Phase 2: Tool Discovery (`session.list_tools()`)

The client asks: *"What tools do you have?"*  
The server responds with a full **JSON Schema** for each tool — name, description, argument types, required fields.

🔑 **This is the crux of MCP.** The client didn't know about `add` before connecting.  
The schema was auto-generated from the server's Python function signature:
```python
@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two numbers together"""
```
Look for `inputSchema` in the server's response — that's the JSON Schema.

In [5]:
async def phase2_tool_discovery():
    async with stdio_client(server_params, errlog=subprocess.DEVNULL) as (read, write):
        logged_read  = LoggingReadStream(read)
        logged_write = LoggingWriteStream(write)

        async with ClientSession(logged_read, logged_write) as session:
            await session.initialize()
            print("(Handshake done — messages above omitted for brevity)\n")
            print("=" * 60)
            print("TOOL DISCOVERY: session.list_tools()")
            print("=" * 60 + "\n")

            tools_result = await session.list_tools()

            print("=" * 60)
            print("✅ Tools discovered:")
            for tool in tools_result.tools:
                print(f"   🔧 {tool.name}: {tool.description}")
                print(f"      inputSchema: {json.dumps(tool.inputSchema, indent=6)}")
            print("=" * 60)

asyncio.run(phase2_tool_discovery())

📤 CLIENT → SERVER:
{
  "method": "initialize",
  "params": {
    "protocolVersion": "2025-11-25",
    "capabilities": {},
    "clientInfo": {
      "name": "mcp",
      "version": "0.1.0"
    }
  },
  "jsonrpc": "2.0",
  "id": 0
}

📥 SERVER → CLIENT:
{
  "jsonrpc": "2.0",
  "id": 0,
  "result": {
    "protocolVersion": "2025-11-25",
    "capabilities": {
      "experimental": {},
      "prompts": {
        "listChanged": false
      },
      "resources": {
        "subscribe": false,
        "listChanged": false
      },
      "tools": {
        "listChanged": false
      }
    },
    "serverInfo": {
      "name": "Calculator",
      "version": "1.27.0"
    }
  }
}

📤 CLIENT → SERVER:
{
  "method": "notifications/initialized",
  "jsonrpc": "2.0"
}

(Handshake done — messages above omitted for brevity)

TOOL DISCOVERY: session.list_tools()

📤 CLIENT → SERVER:
{
  "method": "tools/list",
  "jsonrpc": "2.0",
  "id": 1
}

📥 SERVER → CLIENT:
{
  "jsonrpc": "2.0",
  "id": 1,
  "result": {
  

---
## Cell 6 — Phase 3: Tool Invocation (`session.call_tool()`)

The client calls the `add` tool with `{"a": 2, "b": 3}`.  

Watch the JSON-RPC:
- Client sends a `tools/call` request with the method name and arguments
- Server executes the Python function and returns a `content` array
- Client unpacks `content[0]` as a `TextContent` object

The `TextContent` type check (`isinstance(content, TextContent)`) exists because MCP can return  
different content types — text, images, embedded resources, etc.

In [11]:
async def phase3_tool_invocation():
    async with stdio_client(server_params, errlog=subprocess.DEVNULL) as (read, write):
        logged_read  = LoggingReadStream(read)
        logged_write = LoggingWriteStream(write)

        async with ClientSession(logged_read, logged_write) as session:
            await session.initialize()
            await session.list_tools()
            print("(Handshake + tool discovery done — messages above omitted)\n")
            print("=" * 60)
            print("TOOL INVOCATION: session.call_tool('add', {a: 2, b: 3})")
            print("=" * 60 + "\n")

            result = await session.call_tool("add", arguments={"a": 2, "b": 3})
            content = result.content[0]

            print("=" * 60)
            if isinstance(content, TextContent):
                print(f"   Response type : TextContent")
                print(f"   Raw text      : {content.text}")
                print(f"\n✅ Final answer  : 2 + 3 = {content.text}")
            print("=" * 60)

asyncio.run(phase3_tool_invocation())

📤 CLIENT → SERVER:
{
  "method": "initialize",
  "params": {
    "protocolVersion": "2025-11-25",
    "capabilities": {},
    "clientInfo": {
      "name": "mcp",
      "version": "0.1.0"
    }
  },
  "jsonrpc": "2.0",
  "id": 0
}

📥 SERVER → CLIENT:
{
  "jsonrpc": "2.0",
  "id": 0,
  "result": {
    "protocolVersion": "2025-11-25",
    "capabilities": {
      "experimental": {},
      "prompts": {
        "listChanged": false
      },
      "resources": {
        "subscribe": false,
        "listChanged": false
      },
      "tools": {
        "listChanged": false
      }
    },
    "serverInfo": {
      "name": "Calculator",
      "version": "1.27.0"
    }
  }
}

📤 CLIENT → SERVER:
{
  "method": "notifications/initialized",
  "jsonrpc": "2.0"
}

📤 CLIENT → SERVER:
{
  "method": "tools/list",
  "jsonrpc": "2.0",
  "id": 1
}

📥 SERVER → CLIENT:
{
  "jsonrpc": "2.0",
  "id": 1,
  "result": {
    "tools": [
      {
        "name": "add",
        "description": "Add two numbers together"

---
## Cell 7 — Full Conversation End-to-End (All 3 Phases)

Watch the **entire JSON-RPC conversation** in one go — handshake, discovery, and invocation.  
This is the complete protocol exchange that happens every time an MCP client connects to a server.

In [12]:
async def full_conversation():
    async with stdio_client(server_params, errlog=subprocess.DEVNULL) as (read, write):
        logged_read  = LoggingReadStream(read)
        logged_write = LoggingWriteStream(write)

        async with ClientSession(logged_read, logged_write) as session:

            # ── Phase 1: Handshake ─────────────────────────────────────
            print("=" * 60)
            print("PHASE 1 — HANDSHAKE: session.initialize()")
            print("=" * 60 + "\n")
            init_result = await session.initialize()
            print(f"Server: {init_result.serverInfo.name} v{init_result.serverInfo.version}\n")

            # ── Phase 2: Tool Discovery ────────────────────────────────
            print("=" * 60)
            print("PHASE 2 — DISCOVERY: session.list_tools()")
            print("=" * 60 + "\n")
            tools_result = await session.list_tools()
            print(f"Found {len(tools_result.tools)} tool(s):\n")
            for tool in tools_result.tools:
                print(f"  🔧 {tool.name}: {tool.description}")

            # ── Phase 3: Tool Call ─────────────────────────────────────
            print("\n" + "=" * 60)
            print("PHASE 3 — INVOCATION: session.call_tool('add', {a:2, b:3})")
            print("=" * 60 + "\n")
            result = await session.call_tool("add", arguments={"a": 2, "b": 3})
            content = result.content[0]
            if isinstance(content, TextContent):
                print(f"\n🎉 2 + 3 = {content.text}")

asyncio.run(full_conversation())

PHASE 1 — HANDSHAKE: session.initialize()

📤 CLIENT → SERVER:
{
  "method": "initialize",
  "params": {
    "protocolVersion": "2025-11-25",
    "capabilities": {},
    "clientInfo": {
      "name": "mcp",
      "version": "0.1.0"
    }
  },
  "jsonrpc": "2.0",
  "id": 0
}

📥 SERVER → CLIENT:
{
  "jsonrpc": "2.0",
  "id": 0,
  "result": {
    "protocolVersion": "2025-11-25",
    "capabilities": {
      "experimental": {},
      "prompts": {
        "listChanged": false
      },
      "resources": {
        "subscribe": false,
        "listChanged": false
      },
      "tools": {
        "listChanged": false
      }
    },
    "serverInfo": {
      "name": "Calculator",
      "version": "1.27.0"
    }
  }
}

📤 CLIENT → SERVER:
{
  "method": "notifications/initialized",
  "jsonrpc": "2.0"
}

Server: Calculator v1.27.0

PHASE 2 — DISCOVERY: session.list_tools()

📤 CLIENT → SERVER:
{
  "method": "tools/list",
  "jsonrpc": "2.0",
  "id": 1
}

📥 SERVER → CLIENT:
{
  "jsonrpc": "2.0",
  "id"